# OMNOM TELEPORTATION
# smart unrecorded scheme


Implementation of the Map Equation with the smart unrecorded teleportation 
scheme that the paper actually recommends.

1. **Smart teleportation**:
"Where should I restart the walker so that it can naturally continue the real flow of the network?"
instead of jumping to any node uniformly, the random walker teleports to nodes that have stronger out-going connections, proportional to their out-strength. 

3. **Unrecorded**: when teleportation is unrecorded, teleportation steps in the description length calculation are not counted, so only steps along actual edges are recorder.

This way the results are much less dependent on the arbitrary teleportation parameter τ.
So ofr the result when we use pagerank, what we do is:
- use teleportation to compute an intermediate distribution (p* with smart teleportation),
- then take one extra link-only step to get the "recorded" visit rates (p) that actually go into the formula.

## Validation 

- compare against the official `infomap` package from mapequation.org (the authors' reference). That package uses the same flow model as this implementation. MIND YOU that igraph's `community_infomap` supposedly wraps the same underlying library with some kind of sorcery its optimizer finds different partitions so i use its partition for the official infomap L too and compare that.

In [2]:
import igraph as ig
import numpy as np
import warnings

In [3]:
pip install infomap

Note: you may need to restart the kernel to use updated packages.


In [4]:
# apparently i still need this
def safe_xlogx(x):
    return np.where(x > 0, x * np.log2(x), 0.0)

In [5]:
def pagerank_nonuniform(M, tau=0.15, tol=1e-15, maxiter=int(1e6)):
    """Two-step PageRank for smart unrecorded teleportation.
    Step 1: solve for p* with teleportation proportional to out-strength (eq. 4 from pdf).
    Step 2: one extra link-only step to get the recorded visit rates p (eq. 6 from pdf).
    """
    N = M.shape[0]
    row_sums = M.sum(axis=1) #total weight of edges of node i (out-strngth)
    col_sums = M.sum(axis=0) #total weight coming in the node (in-strngth)
    dangling = (row_sums == 0) #dangling nodes have no outgoing edges
    no_incoming = (col_sums == 0) #no-incoming nodes have nothing pointing at them, walker only reaches them by teleportation
    row_sums_safe = np.where(dangling, 1, row_sums) # normalizing the rows
    M_norm = M / row_sums_safe[:, None] # safe version avoids dev. by 0 for dangling nodes

    # teleportation distribution proportional to out-strength
    total_out = row_sums.sum()
    d = row_sums / total_out if total_out > 0 else np.ones(N) / N #d[i] prob. of teleporting to node i

    # Step 1: solve for p*
    p_star = np.ones(N) / N
    for _ in range(int(maxiter)):
        dangling_sum = p_star[dangling].sum()
        p_star_new = (1 - tau) * (p_star @ M_norm + dangling_sum * d) + tau * d
        if np.linalg.norm(p_star_new - p_star) < tol:
            p_star = p_star_new
            break
        p_star = p_star_new

    # Step 2: extra link-only step + dangling redistribution
    dangling_sum = p_star[dangling].sum()
    p = p_star @ M_norm + dangling_sum * d

    # Nodes with no incoming edges can only be reached via teleportation,
    # which isn't recorded, so their recorded flow is exactly 0
    p[no_incoming] = 0
    p = p / p.sum()
    return p

**Why do we use out-strength and not in-strength?**
- We are trying to simulate restarting the walker by injecting _her_ at node i in a way that doesnt distort the flow structure. This translates to "how useful is that node for producing realistic network flow afterwards?" => computationally: how much the node can distribute flow with out-strength.

So teleportation is used to inject walkers into nodes that behave like sources / distributors of flow. IF we used in-strength the walker would teleport mostly into nodes that collect flow so: 
- these are often sinks or bottlenecks
- once the walker lands there, she may: get stuck (no outgoing edges), or more teleportation

=> basically bias the process toward dead ends of dynamics.

In [6]:
def compute_exit_flow_nonuniform(g, communities, p):
    """Rate of flow leaving each community via outgoing edges."""
    communities = np.array(communities)
    out_strength = np.array(g.strength(mode="out", weights="weight" if g.is_weighted() else None), dtype=float)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src, trg = edges[:, 0], edges[:, 1]
    src_com, trg_com = communities[src], communities[trg]
    betw = src_com != trg_com

    out_str_safe = np.where(out_strength > 0, out_strength, 1.0)
    flow = p[src] * weights / out_str_safe[src]

    exit_flow = np.zeros(max(communities) + 1)
    np.add.at(exit_flow, src_com[betw], flow[betw])
    return exit_flow

In [7]:
def compute_enter_flow_nonuniform(g, communities, p):
    """Rate of flow entering each community via incoming edges from outside."""
    communities = np.array(communities)
    out_strength = np.array(g.strength(mode="out", weights="weight" if g.is_weighted() else None), dtype=float)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src, trg = edges[:, 0], edges[:, 1]
    src_com, trg_com = communities[src], communities[trg]
    betw = src_com != trg_com

    out_str_safe = np.where(out_strength > 0, out_strength, 1.0)
    flow = p[src] * weights / out_str_safe[src]

    enter_flow = np.zeros(max(communities) + 1)
    np.add.at(enter_flow, trg_com[betw], flow[betw])
    return enter_flow

In [8]:
def compute_exit_weights(g, communities):
    communities = np.array(communities)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src_com = communities[edges[:, 0]]
    trg_com = communities[edges[:, 1]]
    betw = src_com != trg_com

    exit_weights = np.zeros(max(communities) + 1)
    np.add.at(exit_weights, src_com[betw], weights[betw])
    np.add.at(exit_weights, trg_com[betw], weights[betw])
    return exit_weights

In [9]:
def compute_description_length(g, communities, tau=0.15):
    """Map equation L using smart unrecorded teleportation for directed graphs,
    and the standard undirected formula for undirected graphs."""
    communities = np.array(communities)
    num_comms = max(communities) + 1
    N = g.vcount()

    if g.is_directed():
        adj = np.array(g.get_adjacency(attribute="weight" if g.is_weighted() else None).data, dtype=float)
        p = pagerank_nonuniform(adj, tau=tau)

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        exit_flow = compute_exit_flow_nonuniform(g, communities, p)
        enter_flow = compute_enter_flow_nonuniform(g, communities, p)

        # enter flow for the index codebook, exit flow for the module codebook
        q_enter = enter_flow
        q_exit = exit_flow
    else:
        weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
        total_weight_x2 = 2 * np.sum(weights)
        strength = np.array(g.strength(weights="weight" if g.is_weighted() else None), dtype=float)
        p = strength / total_weight_x2

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        # for undirected, enter and exit are equal by symmetry
        exit_weights = compute_exit_weights(g, communities)
        q_enter = exit_weights / total_weight_x2
        q_exit = q_enter

    q_enter_sum = np.sum(q_enter)
    p_loop = p_mod + q_exit

    L = safe_xlogx(q_enter_sum) - np.sum(safe_xlogx(q_enter)) \
        - np.sum(safe_xlogx(q_exit)) \
        - np.sum(safe_xlogx(p)) + np.sum(safe_xlogx(p_loop))
    return L

In [10]:
# sanity check 
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")
communities = g.community_infomap()
print("--- unweighted + undirected ---")
print(f"Custom L: {compute_description_length(g, communities.membership):.6f}")
print(f"igraph L: {communities.codelength:.6f}")

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_undirected.graphml")
communities = g.community_infomap(edge_weights="weight")
print("\n--- weighted + undirected ---")
print(f"Custom L: {compute_description_length(g, communities.membership):.6f}")
print(f"igraph L: {communities.codelength:.6f}")

--- unweighted + undirected ---
Custom L: 3.401583
igraph L: 3.401583

--- weighted + undirected ---
Custom L: 3.336712
igraph L: 3.336712


## Validation: comparison against official infomap

The fair test is to evaluate L on the **same partition** with both implementations and check if the formulae agree. Here: take the partition igraph found, then compute L on it with our formula and with the official infomap package (forced onto that same partition). If the two L values match, our formula is validated against the reference.

SO: my custom L ≈ official L (same partition, same formula). igraph's reported L is different because igraph's optimizer found a different partition.

In [11]:
from infomap import Infomap

def L_official_on_partition(g, partition, weighted):
    """Run official Infomap forced onto a given partition and return its L."""
    im = Infomap(directed=True, two_level=True, silent=True)
    for e in g.es:
        im.add_link(e.source, e.target, e["weight"] if weighted else 1.0)
    im.run(initial_partition={node_id: mod for node_id, mod in enumerate(partition)})
    return im.codelength


# unweighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
communities = g.community_infomap() 
partition = communities.membership  

print("--- unweighted + directed ---")
print(f"Custom L:   {compute_description_length(g, partition):.6f}")
print(f"Official L: {L_official_on_partition(g, partition, weighted=False):.6f}")
print(f"igraph L:   {communities.codelength:.6f}")

# weighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
communities = g.community_infomap(edge_weights="weight")
partition = communities.membership
print("\n--- weighted + directed ---")
print(f"Custom L:   {compute_description_length(g, partition):.6f}")
print(f"Official L: {L_official_on_partition(g, partition, weighted=True):.6f}")
print(f"igraph L:   {communities.codelength:.6f}")

--- unweighted + directed ---
Custom L:   3.089377
Official L: 3.121512
igraph L:   3.515830

--- weighted + directed ---
Custom L:   2.736462
Official L: 2.713013
igraph L:   3.392047


C:\Users\savin\AppData\Local\Temp\ipykernel_17664\3885557958.py:3: RuntimeWarning: divide by zero encountered in log2
  return np.where(x > 0, x * np.log2(x), 0.0)
C:\Users\savin\AppData\Local\Temp\ipykernel_17664\3885557958.py:3: RuntimeWarning: invalid value encountered in multiply
  return np.where(x > 0, x * np.log2(x), 0.0)


# DIAGNOSTIC: 
## comparing our PageRank per-node visit rates against official infomap

used this to debug my initial code; just keeping it here as evidence of the investigation + a way to verify the implementation per-node.

In [12]:
# Creates an Infomap object (the official reference implementation), 
# feeds in the same graph we re working with, runs the algorithm.
# im.nodes is a list of all nodes in the graph along with various stats.
# The .flow attribute is the recorded visit rate for that node
# We loop through im.nodes, pick out the flow for each one, and store it in a numpy array official_p indexed by node ID.

# after this, im contains the partition official infomap found and the visit rate for each node
im = Infomap(directed=True, two_level=True, silent=True)
for e in g.es:
    im.add_link(e.source, e.target, e["weight"])
im.run()

# get official Infomap's p (flow per node)
official_p = np.zeros(g.vcount())
for node in im.nodes:
    official_p[node.node_id] = node.flow
# how often the random walker lands on node v according to official infomap

# compute my p on the same graph; our code's estimate of the visit rate for node v
adj = np.array(g.get_adjacency(attribute="weight").data, dtype=float)
my_p = pagerank_nonuniform(adj, tau=0.15)

# obvs theyre distributions so all sums should be 1, but originally mine was ~0.96 (=> dangling node leak)
print("Official p sum:", official_p.sum())
print("My p sum:      ", my_p.sum())
print("Max absolute difference:", np.max(np.abs(official_p - my_p)))
print("Per-node comparison:")
for v in range(min(10, g.vcount())):
    print(f"  node {v}: official={official_p[v]:.6f}  mine={my_p[v]:.6f}  diff={official_p[v]-my_p[v]:+.6f}")

Official p sum: 0.9999999999999999
My p sum:       0.9999999999999999
Max absolute difference: 0.005132828469150502
Per-node comparison:
  node 0: official=0.022232  mine=0.021450  diff=+0.000781
  node 1: official=0.022319  mine=0.021964  diff=+0.000355
  node 2: official=0.022997  mine=0.022944  diff=+0.000053
  node 3: official=0.010201  mine=0.012305  diff=-0.002104
  node 4: official=0.017016  mine=0.017223  diff=-0.000208
  node 5: official=0.018320  mine=0.017974  diff=+0.000347
  node 6: official=0.000000  mine=0.000000  diff=+0.000000
  node 7: official=0.016999  mine=0.018750  diff=-0.001752
  node 8: official=0.015751  mine=0.017376  diff=-0.001625
  node 9: official=0.036015  mine=0.036293  diff=-0.000277


In [13]:
# with this i spotted the pattern that nodes with in_strength = 0 got nonzero p in my code but zero in official
# and that nodes with in > out were systematically underestimated

# show ALL nodes sorted by absolute difference
diffs = official_p - my_p
sorted_indices = np.argsort(-np.abs(diffs))[:20]   # top 20 worst

print(f"{'node':>5}  {'official':>10}  {'mine':>10}  {'diff':>10}  {'out_strength':>12}  {'in_strength':>11}")
for v in sorted_indices:
    out_s = g.strength(v, mode="out", weights="weight")
    in_s = g.strength(v, mode="in", weights="weight")
    print(f"{v:5d}  {official_p[v]:10.6f}  {my_p[v]:10.6f}  {diffs[v]:+10.6f}  {out_s:12.4f}  {in_s:11.4f}")

 node    official        mine        diff  out_strength  in_strength
   13    0.194633    0.189500   +0.005133        1.2558       1.4065
   12    0.101145    0.097948   +0.003198        0.2625       1.6314
   18    0.012425    0.015551   -0.003126        2.6213       0.8903
   15    0.132170    0.129180   +0.002990        1.2173       1.5449
   14    0.091885    0.089262   +0.002622        0.4463       1.7490
    3    0.010201    0.012305   -0.002104        1.8118       0.2427
   24    0.023577    0.025378   -0.001802        1.9352       0.7005
    7    0.016999    0.018750   -0.001752        1.7284       1.5695
    8    0.015751    0.017376   -0.001625        1.6030       1.4142
   17    0.010091    0.011541   -0.001449        1.3273       0.9278
   23    0.026248    0.027428   -0.001179        1.5464       1.0923
   20    0.008761    0.009846   -0.001085        1.0249       0.9151
   19    0.023472    0.022647   +0.000825        0.0000       1.1359
   16    0.004359    0.005183   -0

In [14]:
# lists which nodes have no incoming edges, then prints both implementations' p values for those nodes
# now both show 0 so fixed
# this node has zero contribution to the encoded random walk

in_strengths = np.array(g.strength(mode="in", weights="weight"))
no_incoming = in_strengths == 0

print("Nodes with no incoming edges:", np.where(no_incoming)[0].tolist())
print("Official p for these nodes:  ", official_p[no_incoming])
print("My p for these nodes:        ", my_p[no_incoming])

Nodes with no incoming edges: [6, 10]
Official p for these nodes:   [0. 0.]
My p for these nodes:         [0. 0.]


## 💥 the rule

- Whenever a node receives more flow than it sends out (in > out), official gives it more weight than mine does + whenever a node sends out more flow than it receives (out > in), official gives it less weight than mine does.

- so:
The official infomap is doing something with the in-strength of nodes that im not. Maybe in their internal calculation they weight the visit rate by in-strength somehow, or there's a normalization step that depends on incoming flow. My code only uses out-strength explicitly (in the teleportation distribution d).

## 🔨 try 14/05

"For directed networks, a teleportation rate too close to 0 gives results that depend on how the random
walker was initiated and should be avoided, but a teleportation value equal to 1 corresponds to using the link weights as the
stationary distribution."

In [22]:
# Test the τ=1 case
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
adj = np.array(g.get_adjacency(attribute="weight").data, dtype=float)

my_p_tau1 = pagerank_nonuniform(adj, tau=1.0)

# they say it should be: in-strength normalized
# should theoretically be (just in-strength / total)
in_strength = adj.sum(axis=0)
expected_p = in_strength / in_strength.sum()

print("My p at tau=1:    ", my_p_tau1[:6])
print("Expected (in-str):", expected_p[:6])
print("Max diff:         ", np.max(np.abs(my_p_tau1 - expected_p)))

My p at tau=1:     [0.01730599 0.04420492 0.05807208 0.00879871 0.00921337 0.0212565 ]
Expected (in-str): [0.01730599 0.04420492 0.05807208 0.00879871 0.00921337 0.0212565 ]
Max diff:          1.3877787807814457e-17


so bug in:
link-following / iteration / normalization